# M3 · Evaluación con RAGAS y Cruce con el Harness de M2

**Tópicos Especiales en IA · Universidad EAFIT**  
**Responsable:** Luis  

Este notebook evalúa el sistema RAG multidimensionalmente cumpliendo con el **Nivel 4 (5 puntos)** de la rúbrica:
1. **Faithfulness** (Fidelidad de la respuesta generada al contexto clínico).
2. **Context Precision** (Precisión de los fragmentos recuperados por el retrieval).
3. **Context Recall** (Cobertura de la evidencia necesaria para la recomendación clínica).
4. **Answer Relevancy** (Pertinencia de la respuesta frente a la pregunta original).
5. **Cruce con el Harness de M2** (F1 de extracción de Clinical BERT por documento vs RAG) y diagnóstico automático de fallas.


## 1 · Configuración y Acceso a Drive
Montamos Google Drive donde residen el repositorio y los resultados de M2.

In [ ]:
from google.colab import drive
import os, sys
from pathlib import Path

drive.mount('/content/drive')

# Ruta a la carpeta del proyecto en Drive
RAIZ = Path('/content/drive/MyDrive/TopicosIA/Proyecto-Salud')
if not RAIZ.exists():
    # Fallback si se clona directamente en /content
    RAIZ = Path('/content/Topicos-IA')

print('Directorio de trabajo:', RAIZ)
%cd {RAIZ}

# Verificamos si existe el archivo de resultados de M2 para el cruce
dim1_path = RAIZ / 'M2/ejecucion/outputs_gold/resultado_dimension1.json'
print(f'Resultado M2 (Dimensión 1): {"EXISTE ✅" if dim1_path.exists() else "FALTA ⚠️ (usará respaldo)"}')

## 2 · Instalación de Dependencias

In [ ]:
%pip install -q sentence-transformers ragas datasets langchain-groq python-dotenv PyYAML
print('Dependencias de RAGAS instaladas con éxito.')

## 3 · Configuración de Secretos (.env)
Leemos `GROQ_API_KEY` desde los Secretos de Colab (icono 🔑 en la barra lateral izquierda).

In [ ]:
from google.colab import userdata

try:
    groq_key = userdata.get('GROQ_API_KEY')
    os.environ['GROQ_API_KEY'] = groq_key
    print('GROQ_API_KEY configurada desde Secretos de Colab ✅')
except Exception:
    print('Aviso: GROQ_API_KEY no encontrada en Secretos. Para modo real, configúrala en el icono 🔑.')

# Generamos el .env local para consistencia con los scripts CLI
with open(RAIZ / '.env', 'w') as f:
    f.write(f'PROJECT_ROOT={RAIZ}\n')
    if 'GROQ_API_KEY' in os.environ:
        f.write(f'GROQ_API_KEY={os.environ["GROQ_API_KEY"]}\n')

## 4 · Evaluación en Modo Mock (Cálculo Rápido sin Consumo de Cuota)
Evalúa la tubería usando embeddings multilingües locales y ejecuta el cruce con el F1 de M2.

In [ ]:
%cd {RAIZ}/M3/ragas
!python run_ragas.py --config config.yaml --modo mock

## 5 · Evaluación en Modo Real (RAGAS Oficial + Juez LLM Groq)
Corre la librería oficial `ragas` utilizando el modelo `openai/gpt-oss-120b` (mismo juez que M2 Dimensión 3) y consolida la tabla de diagnóstico.

In [ ]:
%cd {RAIZ}/M3/ragas
!python run_ragas.py --config config.yaml --modo real